# BYTE-PAIR ENCODING BY HAND

Practice the lesson's pipeline on new data: a short list of recipe sentences, then a corpus of DNA reads. You will write the pair counter, the training loop, the encoder and the decoder yourself, then measure how vocabulary size changes sequence length.

***Summary***
1. [Counting pairs](#counting-pairs)
2. [The training loop](#training-loop)
3. [Encoding and decoding](#encoding-decoding)
4. [Vocabulary size against sequence length](#vocab-vs-length)
5. [Generalization: a different alphabet](#generalization)

Run the setup cell below first. It defines the datasets and a few helpers you may call. Do not edit it.

In [ ]:
import os
import re
from collections import Counter

import matplotlib.pyplot as plt

os.makedirs('images', exist_ok=True)

# Provided data and helpers - don't change these lines.
END = '</w>'  # end-of-word marker appended to every word
UNK = '<unk>'  # id used for any piece that is not in the vocabulary

RECIPE_TEXT = (
    'simmer the tomato sauce then stir the sauce and add the basil '
    'roast the potato and the tomato then mash the potato with butter '
    'whisk the butter into the sauce and taste the sauce '
    'slice the onion and the potato then roast the onion '
    'stir the basil into the tomato sauce and serve'
)
HELD_OUT = 'whisk the sauce in the saucepan then roast the tomatoes'


def normalize(text):
    return re.findall(r'[a-z]+', text.lower())


def to_symbols(word):
    return tuple(word) + (END,)


def build_corpus(text):
    # {symbol tuple: number of times the word occurs}
    return {to_symbols(w): n for w, n in Counter(normalize(text)).items()}


def merge_symbols(symbols, pair):
    # Fuse every adjacent occurrence of `pair` in one word, left to right.
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)


def build_vocab(text, merges):
    base = sorted(set(''.join(normalize(text))) | {END})
    symbols = [UNK] + base + [pair[0] + pair[1] for pair, _ in merges]
    vocab = {}
    for sym in symbols:
        vocab.setdefault(sym, len(vocab))
    return vocab


def to_ids(pieces, vocab):
    return [vocab.get(p, vocab[UNK]) for p in pieces]


***
<a id='counting-pairs'></a>
## 1. Counting pairs

Every merge starts from pair counts, so a wrong count here corrupts the whole tokenizer.

**Q1 a) A teammate looks at the recipe corpus and guesses that the pair (t, h) is the most frequent, because the is the commonest word. Check the guess by counting. Write `pair_counts(corpus)` so that it returns a `Counter` mapping each adjacent pair of symbols to its total count, weighting each word by how often it occurs. Store the counts for the starting corpus in a variable named `counts`, then say in the answer cell whether the guess holds.**

Expected output:

`counts` is a `Counter`, and `counts.most_common(5)` lists five (pair, count) entries with the largest count first. Every pair is a tuple of two strings, and pairs that include the end marker `</w>` are counted too.

*Hint:* Use [`zip`](https://docs.python.org/3/library/functions.html#zip) over a word and the same word shifted by one, and [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) to accumulate.

In [ ]:
corpus = build_corpus(RECIPE_TEXT)

### YOUR CODE HERE ###
def pair_counts(corpus):
    ...

counts = ...

#print(counts.most_common(5))


*ANSWER HERE*

**Q1 b) Two pairs can tie for the top count, and without a tie rule two runs of the training code may give different merge tables. Write `best_pair(counts)` so that it returns the pair with the highest count and, when counts tie, the alphabetically first pair. Store the winner for the starting corpus in a variable named `first_pair`.**

Expected output:

`first_pair` is a tuple of two strings. Running the cell twice gives the same tuple, and `counts[first_pair]` equals the maximum of `counts.values()`.

*Hint:* Use [`min`](https://docs.python.org/3/library/functions.html#min) with a `key` function that sorts by negative count first and by the pair itself second.

In [ ]:
### YOUR CODE HERE ###
def best_pair(counts):
    ...

first_pair = ...

#print(first_pair, counts[first_pair])


***
<a id='training-loop'></a>
## 2. The training loop

The ordered list of merges is the tokenizer, so the loop must record each merge in the order it was learned.

**Q2 a) Repeating count, pick and merge by hand twelve times is slow and error-prone, and a sloppy loop crashes when the corpus runs out of pairs. Write `learn_merges(text, num_merges)` that builds the corpus, then repeats: count pairs, take the best pair, record `(pair, count)`, and apply the merge to every word. It must stop early, without an error, if no pairs are left. Store the result of training 12 merges on `RECIPE_TEXT` in a variable named `recipe_merges`.**

Expected output:

`recipe_merges` is a list of at most 12 entries of the form `((left, right), count)`, in the order learned. The first entry's pair equals your `first_pair` from Q1 b.

*Hint:* Reuse your `pair_counts` and `best_pair`, and the provided `merge_symbols`. Rebuild the corpus dictionary after each merge, and add the frequencies if two words end up with the same symbol tuple. A [`while` loop with `break`](https://docs.python.org/3/tutorial/controlflow.html#break-and-continue-statements-and-else-clauses-on-loops) handles the early stop.

In [ ]:
### YOUR CODE HERE ###
def learn_merges(text, num_merges):
    ...

recipe_merges = ...

#print(len(recipe_merges), recipe_merges[:3])


***
<a id='encoding-decoding'></a>
## 3. Encoding and decoding

Encoding replays the merges on text the tokenizer has never seen, and decoding must undo it.

**Q3 a) The held-out sentence contains saucepan and tomatoes, which are not in the training text, yet the tokenizer must not fail on them. Write `encode(text, merges)` that returns a flat list of pieces for the whole text, applying the merges to each word in the order they were learned. Store the pieces for `HELD_OUT` in a variable named `held_out_pieces`, then say in the answer cell where saucepan splits and why.**

Expected output:

`held_out_pieces` is a list of strings. Every word in the sentence contributes pieces, and the last piece of each word ends with `</w>`.

*Hint:* Use [`re.findall`](https://docs.python.org/3/library/re.html#re.findall) through the provided `normalize`, then loop over the merges for each word.

In [ ]:
### YOUR CODE HERE ###
def encode(text, merges):
    ...

held_out_pieces = ...

#print(held_out_pieces)


*ANSWER HERE*

**Q3 b) Decoded sentences come back with the end markers glued in, like `whisk</w>the</w>sauce</w>`. Write `decode(ids, vocab)` that turns ids back into text by looking each id up, joining the pieces, and turning the end marker back into a space. Store the decoded `ids` in a variable named `restored`.**

Expected output:

`restored` is a string with single spaces between words and no `</w>` in it. It equals the lowercase, punctuation-free form of `HELD_OUT`.

*Hint:* Invert the `vocab` dictionary once with a [dict comprehension](https://docs.python.org/3/tutorial/datastructures.html#dictionaries), then use `str.join` and `str.replace`.

In [ ]:
vocab = build_vocab(RECIPE_TEXT, recipe_merges)
ids = to_ids(held_out_pieces, vocab)

### YOUR CODE HERE ###
def decode(ids, vocab):
    ...

restored = ...

#assert restored == ' '.join(normalize(HELD_OUT))


**Q3 c) A menu item reads jalapeno, and the letter j never appears in the recipe corpus. The pipeline does not crash, but the decoded text no longer matches the input. Count how many unknown ids `jalapeno` produces and store the count in a variable named `unk_count`, then say in the answer cell what design change would avoid the loss.**

Expected output:

`unk_count` is a non-negative integer. Decoding the ids for `jalapeno` does not give back `jalapeno`.

*Hint:* Use your `encode` and the provided `to_ids`, then count the ids equal to `vocab[UNK]` with [`list.count`](https://docs.python.org/3/tutorial/datastructures.html#more-on-lists).

In [ ]:
### YOUR CODE HERE ###
unk_count = ...

#print(encode('jalapeno', recipe_merges), unk_count)


*ANSWER HERE*

***
<a id='vocab-vs-length'></a>
## 4. Vocabulary size against sequence length

This is the measurement a decision maker cares about: how many tokens does the same text cost as the vocabulary grows?

**Q4 a) A manager asks how much shorter the held-out sentence gets if you train more merges. Train once with `learn_merges` for the largest value in `merge_counts`, then for each value n encode `HELD_OUT` using only the first n learned merges. Store the encoded lengths in a list named `lengths` and the matching vocabulary sizes (from `build_vocab`) in a list named `vocab_sizes`.**

Expected output:

`lengths` and `vocab_sizes` each have one entry per value in `merge_counts`. `lengths` never increases from left to right, and `vocab_sizes` never decreases.

*Hint:* Use [list slicing](https://docs.python.org/3/tutorial/introduction.html#lists) to take the first n merges, and a list comprehension for each result.

In [ ]:
merge_counts = [0, 2, 4, 8, 16, 32]  # don't change this line

### YOUR CODE HERE ###

# Name the lists as follows:
lengths = ...
vocab_sizes = ...

#print(list(zip(merge_counts, vocab_sizes, lengths)))


**Q4 b) Plot encoded length against vocabulary size as one line, label both axes, and save it to `images/hw01_vocab_vs_length.png`. Store the matplotlib figure in a variable named `fig`, then say in the answer cell where the curve stops falling and what in this corpus causes that.**

Expected output:

One line that falls from left to right, with axis labels, and a PNG file at `images/hw01_vocab_vs_length.png`.

*Hint:* Use [`matplotlib.pyplot.savefig`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html) on the figure before calling `plt.show()`.

In [ ]:
### YOUR CODE HERE ###
fig = ...

#assert os.path.exists('images/hw01_vocab_vs_length.png')


*ANSWER HERE*

***
<a id='generalization'></a>
## 5. Generalization: a different alphabet

The same pipeline works on any sequence of symbols, not just English words. Here each DNA read plays the role of a word and the alphabet is four letters.

**Q5 a) Apply your training loop to a DNA corpus where each read is a word. Train 10 merges on `DNA_READS` with your `learn_merges` and store the result in a variable named `dna_merges`.**

Expected output:

`dna_merges` is a list of at most 10 `((left, right), count)` entries. Every symbol is built only from the letters a, c, g, t and the end marker `</w>`.

*Hint:* No new code is needed beyond calling the function you already wrote; check that the provided `normalize` keeps the read letters.

In [ ]:
DNA_READS = (
    'gattaca gattaca gattaca gattaca tagaca tagaca cagatta cagatta '
    'gattagaca tagattaca acgtacgt acgtacgt acgtacgt cgtacgta tacgtacg gattacagatta'
)
DNA_HELD_OUT = 'gattaca acgtacgtacgt tagattagaca'  # don't change these lines

### YOUR CODE HERE ###
dna_merges = ...

#print(dna_merges)


**Q5 b) Measure how much the DNA tokenizer shortens the held-out reads. Compute the encoded length of `DNA_HELD_OUT` with zero merges divided by its encoded length with all of `dna_merges`, and store the ratio in a variable named `dna_ratio`. Then say in the answer cell what a ratio of exactly 1.0 would tell you.**

Expected output:

`dna_ratio` is a float of at least 1.0. A value of exactly 1.0 would mean the merges never fired on the held-out reads.

*Hint:* Reuse your `encode` twice, once with an empty list of merges and once with `dna_merges`.

In [ ]:
### YOUR CODE HERE ###
dna_ratio = ...

#print(dna_ratio)


*ANSWER HERE*